# 03 — Modèle ML — Classification du ton médiatique
**Hackathon iSHEERO × DataCamp 2026 — Bénin Insights Challenge**

Objectif : prédire si un événement médiatique sur le Bénin génère un ton positif ou négatif.

Algorithme : Random Forest Classifier (scikit-learn)
Données : `benin_enrichi.csv` — 10 722 événements, 2025

## 1. Imports

In [ ]:
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

## 2. Chargement des données

In [ ]:
ROOT = Path("../")
df = pd.read_csv(ROOT / "data/processed/benin_enrichi.csv")
print(f"Dataset chargé : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")

## 3. Variable cible

Ton binaire : 1 si `AvgTone > 0` (ton positif), 0 sinon.

In [ ]:
df["ton_binaire"] = (df["AvgTone"] > 0).astype(int)

print("Distribution de la cible :")
print(df["ton_binaire"].value_counts().rename({0: "Négatif (0)", 1: "Positif (1)"}).to_string())
print(f"
Ratio positif : {df['ton_binaire'].mean():.1%}")

## 4. Features et préparation

Variables retenues : type d'événement, stabilité, volume médiatique, saison, zone géographique.
 exclue (dérivée de ).

In [ ]:
FEATURES = ["EventRootCode", "QuadClass", "GoldsteinScale",
            "NumMentions", "NumArticles", "mois", "zone_benin"]

df_ml = df[FEATURES + ["ton_binaire"]].dropna()
print(f"Lignes disponibles après dropna : {len(df_ml):,}")

# Encodage one-hot de zone_benin
X = pd.get_dummies(df_ml[FEATURES], columns=["zone_benin"])
y = df_ml["ton_binaire"]

print(f"Features finales : {list(X.columns)}")

## 5. Séparation train / test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train : {len(X_train):,} événements")
print(f"Test  : {len(X_test):,} événements")

## 6. Entraînement — Random Forest

In [ ]:
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print("Modèle entraîné.")

## 7. Évaluation

In [ ]:
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy : {acc:.4f}")
print()
print(classification_report(y_test, y_pred, target_names=["Négatif (0)", "Positif (1)"]))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Négatif", "Positif"])
disp.plot(ax=ax, colorbar=False)
ax.set_title("Matrice de confusion")
plt.tight_layout()
plt.savefig(ROOT / "notebooks/outputs/ml_confusion_matrix.png", dpi=120)
plt.show()

## 8. Importance des variables

In [ ]:
importances = pd.Series(clf.feature_importances_, index=X.columns)
              .sort_values(ascending=False)

print("Top 5 variables les plus importantes :")
print(importances.head(5).round(4).to_string())

fig, ax = plt.subplots(figsize=(7, 4))
importances.head(8).sort_values().plot(kind="barh", ax=ax)
ax.set_title("Importance des variables — Random Forest")
ax.set_xlabel("Importance (Gini)")
plt.tight_layout()
plt.savefig(ROOT / "notebooks/outputs/ml_feature_importance.png", dpi=120)
plt.show()

## 9. Sauvegarde du modèle

In [ ]:
model_path = ROOT / "models/random_forest_ton.pkl"
(ROOT / "models").mkdir(exist_ok=True)
joblib.dump(clf, model_path)
print(f"Modèle sauvegardé : {model_path}")
print(f"Taille : {model_path.stat().st_size / 1024:.0f} Ko")

## 10. Synthèse

| Métrique | Valeur |
|---|---|
| Accuracy | 0.72 |
| F1 Négatif | 0.77 |
| F1 Positif | 0.63 |
| Variable la plus importante |  (0.36) |
| 2e variable |  (0.24) |

**Lecture** : le mois de l'année et le score Goldstein sont les deux meilleurs prédicteurs du ton d'un article sur le Bénin. Le type d'événement () arrive en 3e position — cohérent avec l'EDA (section 2, corrélation QuadClass/AvgTone = −0,42).

Limite : le modèle détecte une tendance structurelle, pas une causalité. Les anomalies de décembre 2025 influencent probablement le poids de .